# TASK 04 — CLIP Image Embedding Extraction & Multi-View Aggregation

**Project:** SecondLife AI Pricing  
**Domain:** Pre-owned Household Appliances  
**Target Architecture:** Multimodal Feature Extraction (PhoBERT Text + CLIP Vision + Tabular Features)  
**Model:** `openai/clip-vit-base-patch32` (Pretrained Vision Transformer Backbone)  
**Objective:** Extract projected 512-dimensional, L2-normalized image embeddings, aggregate multi-view angles per interaction, and validate numerical properties without fine-tuning.  

> **STRICT GOVERNANCE:** No model weights are fine-tuned. Embeddings are extracted using frozen pretrained weights in `torch.inference_mode()`. Brazilian `original_value` is never used as a target price label.

## 1. Environment Setup & Device Configuration

We detect available hardware acceleration (NVIDIA CUDA GPU on Kaggle/Colab, or CPU locally) and set deterministic seeds.

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
from typing import List, Dict, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
from transformers import CLIPModel, CLIPProcessor

# Set project root
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.features.extract_clip_embeddings import ClipFeatureExtractor
from src.features.aggregate_image_embeddings import ImageEmbeddingAggregator

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"Execution Device: {device}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
else:
    print("Running on multi-core CPU. Recommended for Kaggle: switch to GPU T4 x2 accelerator.")

## 2. Dataset Manifest Inspection

We load the verified manifest (`reports/clip_dataset_manifest.csv`) containing 350 curated appliance photographs spanning 312 distinct interactions across 15 categories.

In [ ]:
manifest_path = Path("reports/clip_dataset_manifest.csv")
assert manifest_path.exists(), f"Manifest not found at {manifest_path}"

df_manifest = pd.read_csv(manifest_path)
print(f"Total Image Records: {len(df_manifest):,}")
print(f"Unique Interactions: {df_manifest['interaction_id'].nunique():,}")
print(f"Categories Represented: {df_manifest['category'].nunique():,}")
print("\nCategory distribution:")
print(df_manifest['category'].value_counts())

# Sample preview
df_manifest[['image_id', 'interaction_id', 'category', 'brand', 'view', 'local_path']].head()

## 3. CLIP Feature Extraction (Image-Level 512-d)

We instantiate `ClipFeatureExtractor` which loads `openai/clip-vit-base-patch32`, verifies image integrity, runs batched inference in `torch.inference_mode()`, and applies L2 normalization.

In [ ]:
output_dir = Path("data/embeddings")
output_dir.mkdir(parents=True, exist_ok=True)

extractor = ClipFeatureExtractor(
    model_name="openai/clip-vit-base-patch32",
    device=device,
    batch_size=32,
)

extract_meta = extractor.extract_from_manifest(
    manifest_path=manifest_path,
    output_dir=output_dir,
    batch_size=32,
)

print(json.dumps(extract_meta, indent=2))

## 4. Multi-View Aggregation by Interaction ID

A single physical appliance frequently has multiple angle photographs (e.g. front exterior, open door, interior shelves, serial plate).  
To represent each physical item as a single vector without data leakage:
1. Group embeddings by `interaction_id`.
2. Perform mean pooling across all view vectors.
3. Apply L2 normalization to ensure unit length.

In [ ]:
aggregator = ImageEmbeddingAggregator()
agg_meta = aggregator.aggregate_and_save(
    embeddings_path=output_dir / "clip_image_embeddings.npy",
    mapping_path=output_dir / "clip_image_mapping.csv",
    output_dir=output_dir,
)

print(json.dumps(agg_meta, indent=2))

## 5. Statistical & Numerical Integrity Validation

We verify mathematical invariants across the generated embedding matrices:
- Exact shapes: `(350, 512)` and `(312, 512)`
- No NaN or Inf values
- L2 norms equal 1.0 within $10^{-4}$

In [ ]:
image_embeddings = np.load(output_dir / "clip_image_embeddings.npy")
interaction_embeddings = np.load(output_dir / "clip_interaction_embeddings.npy")
df_image_map = pd.read_csv(output_dir / "clip_image_mapping.csv")
df_interaction_map = pd.read_csv(output_dir / "clip_interaction_mapping.csv")

print("=== Image Embeddings Validation ===")
print(f"Shape: {image_embeddings.shape} (Expected: (350, 512))")
print(f"Dtype: {image_embeddings.dtype}")
print(f"NaNs: {np.isnan(image_embeddings).sum()}, Infs: {np.isinf(image_embeddings).sum()}")
img_norms = np.linalg.norm(image_embeddings, axis=1)
print(f"L2 Norm Min: {img_norms.min():.6f}, Max: {img_norms.max():.6f}, Mean: {img_norms.mean():.6f}")

print("\n=== Interaction Embeddings Validation ===")
print(f"Shape: {interaction_embeddings.shape} (Expected: (312, 512))")
print(f"Dtype: {interaction_embeddings.dtype}")
print(f"NaNs: {np.isnan(interaction_embeddings).sum()}, Infs: {np.isinf(interaction_embeddings).sum()}")
inter_norms = np.linalg.norm(interaction_embeddings, axis=1)
print(f"L2 Norm Min: {inter_norms.min():.6f}, Max: {inter_norms.max():.6f}, Mean: {inter_norms.mean():.6f}")

assert image_embeddings.shape == (350, 512)
assert interaction_embeddings.shape == (312, 512)
assert np.allclose(img_norms, 1.0, atol=1e-4)
assert np.allclose(inter_norms, 1.0, atol=1e-4)
print("\nAll assertions passed successfully!")

## 6. Multi-View Angle Coherence Analysis

We investigate whether different photo angles of the **same physical appliance** exhibit higher cosine similarity than photos of **different appliances**.

In [ ]:
# Compute within-interaction similarity for multi-view appliances
multi_view_rows = df_interaction_map[df_interaction_map['image_count'] > 1]
within_sims = []

for _, row in multi_view_rows.iterrows():
    indices = json.loads(row['source_image_indices'])
    vecs = image_embeddings[indices]
    for i in range(len(vecs)):
        for j in range(i + 1, len(vecs)):
            sim = np.dot(vecs[i], vecs[j])
            within_sims.append(sim)

# Sample random between-interaction similarities
np.random.seed(42)
between_sims = []
sample_indices = np.random.choice(len(image_embeddings), size=(len(within_sims), 2), replace=True)
for idx1, idx2 in sample_indices:
    if df_image_map.iloc[idx1]['interaction_id'] != df_image_map.iloc[idx2]['interaction_id']:
        sim = np.dot(image_embeddings[idx1], image_embeddings[idx2])
        between_sims.append(sim)

print(f"Within-Interaction (Same Appliance, Different Angles) Mean Cosine Similarity: {np.mean(within_sims):.4f} +/- {np.std(within_sims):.4f}")
print(f"Between-Interaction (Different Appliances) Mean Cosine Similarity:           {np.mean(between_sims):.4f} +/- {np.std(between_sims):.4f}")

plt.figure(figsize=(9, 4))
sns.kdeplot(within_sims, label="Within-Appliance Views (Same Unit)", fill=True, color="forestgreen")
sns.kdeplot(between_sims, label="Between-Appliance Views (Different Units)", fill=True, color="crimson")
plt.title("Cosine Similarity Distribution: Same Physical Unit vs Different Appliances")
plt.xlabel("Cosine Similarity")
plt.ylabel("Density")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Category Representation in CLIP Latent Space

We calculate the mean centroid vector for each appliance category and compute the inter-category cosine distance matrix.

In [ ]:
categories = df_interaction_map['category'].unique()
cat_centroids = {}

for cat in categories:
    cat_idx = df_interaction_map[df_interaction_map['category'] == cat]['interaction_embedding_index'].values
    cat_vecs = interaction_embeddings[cat_idx]
    centroid = np.mean(cat_vecs, axis=0)
    centroid = centroid / np.linalg.norm(centroid)
    cat_centroids[cat] = centroid

cat_names = sorted(list(cat_centroids.keys()))
centroid_matrix = np.array([cat_centroids[c] for c in cat_names])
similarity_matrix = np.dot(centroid_matrix, centroid_matrix.T)

plt.figure(figsize=(11, 9))
sns.heatmap(
    similarity_matrix,
    xticklabels=cat_names,
    yticklabels=cat_names,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    cbar_kws={'label': 'Cosine Similarity'}
)
plt.title("Inter-Category CLIP Latent Centroid Cosine Similarity Matrix")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Artifact Summary

The following validated embedding artifacts are persisted for downstream multimodal pricing fusion:
- `data/embeddings/clip_image_embeddings.npy` (350 × 512)
- `data/embeddings/clip_interaction_embeddings.npy` (312 × 512)
- `data/embeddings/clip_image_mapping.csv` (350 rows)
- `data/embeddings/clip_interaction_mapping.csv` (312 rows)

> **Status:** Pretrained visual representations ready for feature concatenation with PhoBERT text embeddings and tabular metadata in TASK 05.